# S5.M11-1 M6 在其余 4 个代理显著任务上的真实 SGB 复核

**目的**：方法筛选第六阶段第 1 步（[DR-M5-017](../../../registry/decision_records/DR-M5-017.yaml) · D-1 ~ D-3）。

报告 v1.7 的默认组合是「I2 + I3 + M6」。M6（6 个 SGB 成员的概率平均）在明文代理上 7 个任务显著（Holm 后），真实 SGB 只复核过其中 3 个：
- FedAds 成立、mock 重合 80 刚好成立（S5.M2 / S5.M4）；
- COIL2000 不显著。

本步补齐其余 4 个：mock 重合 50、mock 平衡、HomeCredit、Santander 交易。

**做法**：
- M6 的配置与 S5.M2 / S5.M4 相同：优化版联邦配置、行采样 0.8，成员 seed 1100 ~ 1105；
- 对照：S5.M9 已拟合的优化版联邦 SGB 打分（同一输入、seed 11）；
- 用主动方明文直方图版本（S5.M7）。每个任务第一个成员前抽查两项：
  - 前 2 棵树原版与明文逐位相同；
  - 按明文前 2 / 4 棵外推单个成员的秒数，超过 7,000 秒的任务不做；
- 成员逐个存档、分段运行，每段约 2 小时，可续跑；
- 对比口径同 S5.M2：测试客户簇 bootstrap 1,000 次。

**数据**：公开数据，属开发证据。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m11_sgb_m6.yaml"
m11 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg = yaml.safe_load((REPO_ROOT / m11["base_config"]).read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("sgb_m6_more.py", "sgb_i2i3_confirm.py", "sgb_cost_screen.py", "method_screen_sgb.py", "method_screen.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in [CONFIG_PATH, REPO_ROOT / m11["decision_record"]] + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m11["decision_record"], "| step:", m11["step_id"], "|", m11["purpose"])
print("seed:", m11["seed"], "|", m11["seeds_note"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
M9, M11 = (pathlib.Path(os.environ[m11[k]]) for k in ("opt_dir_env", "out_dir_env"))
launch, n_seg = {}, 0
for line in (M11 / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    if line.startswith("=="):
        n_seg += 1
    elif "sha256:" in line:
        path, digest = line.split(" sha256:")
        launch.setdefault(pathlib.Path(path).name, set()).add(digest)
print("启动指纹：%d 段运行，每段记录的文件与 sha256（路径只留文件名）：" % n_seg)
for name, digests in sorted(launch.items()):
    print("  %s %s" % (name, ", ".join("sha256:" + d[:HASH_PREFIX_LEN] for d in sorted(digests))))
for p in COMPONENTS + [CONFIG_PATH]:
    assert launch[p.name] == {hashlib.sha256(p.read_bytes()).hexdigest()}, p.name
print("各段运行时的组件与配置与当前仓库逐字节相同（决策记录在运行后追加了复核，sha256 以上一行为准）。")

modules/m5_modeling/configs/s5_m11_sgb_m6.yaml sha256:1767db41c890
registry/decision_records/DR-M5-017.yaml sha256:2fb9cef48e69
modules/m5_modeling/components/sgb_m6_more.py sha256:a1d86a6ba022
modules/m5_modeling/components/sgb_i2i3_confirm.py sha256:7b2d6ed9a322
modules/m5_modeling/components/sgb_cost_screen.py sha256:cf6088aa95be
modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
决策记录: registry/decision_records/DR-M5-017.yaml | step: S5.M11 | M6 在 mock 平衡、mock 重合 50、HomeCredit、Santander 交易上的真实 SGB，与同配置代理之差、SGB 上的 M6 − 优化版联邦
seed: 11 | M6 的 6 个成员 seed 为 1100 ~ 1105（method_screen_sgb.bag_seeds，同 S5.M2 / S5.M4）；对照的优化版联邦为 seed 11 单次拟合（不含行采样时为确定性拟合）；6 个 seed 只用于读取 S5.M1 的代理结果
git: a5cd05e（工作区含本步未提交改动）
启动指纹：6 段运行，每段记录的文件与 sha256（路径只留文件名）：
  DR-M5-017.yaml sha256:d772990c59be
  method_screen.py sha256:05e19e025d0b
  method_screen_sgb.py sha256:2e54c618305f
  s5_m11_sgb_m6.yaml sha256:1767db41c890
  sgb

## 1. 运行概况

下一格列出每个任务的抽查与拟合信息：
- 前 2 棵逐位抽查的结果；
- 单个成员的外推秒数；
- 6 个成员的实际拟合秒数合计。

In [2]:
from modules.m5_modeling.components import sgb_m6_more as M6

P = M6.MS.Params(cfg)
KEY = "%s − %s" % M6.PAIR
summary = json.loads((M11 / "summary.json").read_text(encoding="utf-8"))
EVAL = {n: json.loads((M11 / ("%s__F_M6__eval.json" % n)).read_text(encoding="utf-8")) for n in m11["tasks"]
        if (M11 / ("%s__F_M6__eval.json" % n)).exists()}
rows = []
for n in m11["tasks"]:
    pr = json.loads((M11 / ("%s__F_M6__probe.json" % n)).read_text(encoding="utf-8"))
    ev = EVAL.get(n, {})
    rows.append({"任务": n, "前 2 棵逐位一致": pr["identical_first_trees"], "单个成员外推秒数": pr["projected_member_seconds"],
                 "6 个成员合计秒数": ev.get("fit_seconds_total"), "判定": pr["verdict"]})
plan = pd.DataFrame(rows)
print("完成对比的任务：%d / %d；判定不做：%s" % (len(EVAL), len(m11["tasks"]), summary["skipped"] or "无"))
print(plan.to_string(index=False))

完成对比的任务：4 / 4；判定不做：无
            任务  前 2 棵逐位一致    单个成员外推秒数  6 个成员合计秒数 判定
mock_overlap50       True  523.683197     3268.4 ok
 mock_balanced       True 1317.407385     7219.5 ok
    homecredit       True 1250.105533     7833.1 ok
  santander_tx       True 2704.588973    15938.2 ok


## 2. 复现

每个任务的 6 个成员合计约 1 ~ 4.5 小时，notebook 不重拟合 SGB。下一格在第一个任务（mock 重合 50）上做三件事：
- 用仓库组件重拟合同配置代理（6 个成员）；
- 从存档的成员打分重算 M6 与对比；
- 重算跨任务汇总。

要求与批量运行相同。

In [3]:
REPRO_TOL = 1e-12
NAME = m11["tasks"][0]
res = json.loads((RUN / (NAME + ".json")).read_text(encoding="utf-8"))
sc = M6.MS.Screen(NAME, cfg, PRIV, M11 / "unused")
again = M6.evaluate_task(P, sc, res, M11, M9)
for m in M6.PAIR:
    a, b = again["methods"][m]["sgb_minus_proxy"], EVAL[NAME]["methods"][m]["sgb_minus_proxy"]
    assert all(abs(a[k] - b[k]) <= REPRO_TOL for k in ("diff", "ci_low", "ci_high", "p")), m
c1, c2 = again["contrasts"][KEY]["sgb"], EVAL[NAME]["contrasts"][KEY]["sgb"]
assert all(abs(c1[k] - c2[k]) <= REPRO_TOL for k in ("diff", "ci_low", "ci_high", "p"))
again_sum = M6.summarize(EVAL, summary["skipped"], m11["proxy_tol"])
assert [r["sgb_holds"] for r in again_sum["tasks"]] == [r["sgb_holds"] for r in summary["tasks"]]
print("%s：用仓库组件重拟合同配置代理（6 个成员）、从存档的成员打分重算 M6 与对比，与批量运行相同（差 ≤ %g）；跨任务汇总重算相同。" % (NAME, REPRO_TOL))

mock_overlap50：用仓库组件重拟合同配置代理（6 个成员）、从存档的成员打分重算 M6 与对比，与批量运行相同（差 ≤ 1e-12）；跨任务汇总重算相同。


## 3. 结果

- **SGB 与同配置代理**：M6 与优化版联邦各自的 SGB − 代理，以及差的绝对值是否在 0.005 以内（DR-M5-012 · D-1）。
- **SGB 上的 M6 − 优化版联邦**：配对差、区间与 p，与代理（6 seed）的结论逐任务对照。
- **合并已提交的 3 个任务**：只读已公开的汇总 CSV（S5.M2 的 FedAds、COIL2000；S5.M4 的 mock 重合 80），得到 7 个代理显著任务的 M6 SGB 证据。
- **每个成员**：seed、拟合秒数与单个成员的 SGB 测试 AUROC。
- 汇总 CSV 写入 `results/sgb_m6_more_v1/`。

In [4]:
OUT = REPO_ROOT / m11["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
R, TOL = m11["round_auc"], m11["proxy_tol"]
vs = pd.DataFrame([{"dataset": n, "method": m, "sgb_auroc": v["sgb_auroc"], "proxy_auroc": v["proxy_auroc"],
                    "sgb_minus_proxy": v["sgb_minus_proxy"]["diff"], "ci_low": v["sgb_minus_proxy"]["ci_low"],
                    "ci_high": v["sgb_minus_proxy"]["ci_high"], "within_tolerance": abs(v["sgb_minus_proxy"]["diff"]) <= TOL}
                   for n, ev in EVAL.items() for m, v in ev["methods"].items()])
con = pd.DataFrame([{"dataset": r["task"], "contrast": KEY, "sgb_diff": r["sgb_diff"], "sgb_ci_low": r["sgb_ci_low"],
                     "sgb_ci_high": r["sgb_ci_high"], "sgb_p": r["sgb_p"],
                     "proxy_6seed_diff": r["proxy_diff"], "proxy_6seed_ci_low": EVAL[r["task"]]["contrasts"][KEY]["proxy_6seed"]["ci_low"],
                     "proxy_6seed_ci_high": EVAL[r["task"]]["contrasts"][KEY]["proxy_6seed"]["ci_high"],
                     "sgb_holds": r["sgb_holds"], "proxy_holds": r["proxy_holds"], "agree": r["agree"]} for r in summary["tasks"]])
mem = pd.DataFrame([{"dataset": n, "member": j, **m} for n, ev in EVAL.items() for j, m in enumerate(ev["members"])])
# 与已提交的 3 个任务（S5.M2：FedAds、COIL2000；S5.M4：mock 重合 80）合并成 7 个代理显著任务的 M6 SGB 证据（只读已公开的汇总 CSV）
prev = pd.concat([pd.read_csv(REPO_ROOT / "modules/m5_modeling/results/method_screen_sgb_v1/sgb_contrasts.csv"),
                  pd.read_csv(REPO_ROOT / "modules/m5_modeling/results/method_screen_sgb_m6_mock80_v1/sgb_contrasts.csv")])
prev = prev[prev["contrast"] == KEY].assign(source="已提交（S5.M2 / S5.M4）")
all7 = pd.concat([con.assign(source="S5.M11")[prev.columns.tolist()], prev], ignore_index=True)
all7["sgb_holds"] = all7["sgb_ci_low"] > 0
all7["proxy_holds"] = all7["proxy_6seed_ci_low"] > 0
vs.to_csv(OUT / "sgb_vs_proxy.csv", index=False, float_format="%.5f")
con.to_csv(OUT / "sgb_m6_contrasts.csv", index=False, float_format="%.5f")
mem.to_csv(OUT / "sgb_members.csv", index=False, float_format="%.5f")
plan.rename(columns={"任务": "dataset", "前 2 棵逐位一致": "identical_first_two_trees", "单个成员外推秒数": "projected_member_seconds",
                     "6 个成员合计秒数": "fit_seconds_total", "判定": "verdict"}).to_csv(OUT / "probes.csv", index=False, float_format="%.1f")
all7.to_csv(OUT / "m6_sgb_all7.csv", index=False, float_format="%.5f")
print("SGB 与同配置代理（容差 %.3f）：" % TOL)
print(vs.round(R).to_string(index=False))
print("SGB 上的 M6 − 优化版联邦：")
print(con.round(R).to_string(index=False))
print("M6 在 SGB 上成立 %d / %d；与代理结论一致 %d / %d" % (summary["sgb_holds"], summary["n"], summary["agree"], summary["n"]))
print("合并已提交的 3 个任务（共 %d 个代理显著任务）：SGB 上成立 %d" % (len(all7), all7["sgb_holds"].sum()))
print(all7.round(R).to_string(index=False))
print("每个成员：")
print(mem.round(R).to_string(index=False))

SGB 与同配置代理（容差 0.005）：
       dataset  method  sgb_auroc  proxy_auroc  sgb_minus_proxy  ci_low  ci_high  within_tolerance
mock_overlap50    F_M6     0.9146       0.9126           0.0020 -0.0013   0.0054              True
mock_overlap50 F_opt_p     0.9049       0.9060          -0.0010 -0.0072   0.0053              True
 mock_balanced    F_M6     0.9061       0.9057           0.0004 -0.0024   0.0032              True
 mock_balanced F_opt_p     0.9015       0.8995           0.0020 -0.0027   0.0064              True
    homecredit    F_M6     0.7557       0.7543           0.0014 -0.0002   0.0031              True
    homecredit F_opt_p     0.7519       0.7509           0.0009 -0.0012   0.0032              True
  santander_tx    F_M6     0.8701       0.8700           0.0001 -0.0035   0.0035              True
  santander_tx F_opt_p     0.8586       0.8598          -0.0012 -0.0066   0.0035              True
SGB 上的 M6 − 优化版联邦：
       dataset       contrast  sgb_diff  sgb_ci_low  sgb_ci_high  sg

## 4. 结论

1. **明文抽查与时长**：
   - 4 个任务第一个成员前的前 2 棵，原版与明文打分全部逐位相同；
   - 外推与实际成员拟合秒数之比在 0.90 ~ 1.13 之间，没有拟合超出上限；
   - 分 6 段完成，24 个成员合计拟合约 9.5 小时，其中 Santander 交易约 4.4 小时。
2. **M6 在真实 SGB 上：4 个任务全部成立**（M6 − 优化版联邦，括号内为代理）：
   - mock 重合 50 +0.0097 [+0.0054, +0.0147]（+0.0065）；
   - mock 平衡 +0.0046 [+0.0013, +0.0080]（+0.0065）；
   - HomeCredit +0.0038 [+0.0023, +0.0056]（+0.0036）；
   - Santander 交易 +0.0116 [+0.0081, +0.0152]（+0.0099）。
3. **与代理一致**：4 / 4 结论一致。M6 与优化版联邦的「SGB − 同配置代理」都在 0.005 以内（−0.0012 ~ +0.0020）。
4. **合并已提交的 3 个任务，7 个代理显著任务中 6 个在真实 SGB 上成立**：
   - 已提交的 3 个：FedAds +0.0113 成立、mock 重合 80 +0.0036 成立、COIL2000 +0.0027 不显著；
   - 与代理结论一致 6 / 7，不一致的只有 COIL2000：代理上 +0.008 成立，SGB 上 +0.003、区间含 0。
5. **单个成员**（行采样 0.8，SGB 测试 AUROC）：
   - mock 重合 50、Santander 交易：6 个成员都高于优化版联邦、都低于 6 个成员的平均；
   - mock 平衡：6 个成员中 4 个低于优化版联邦（成员 0.898 ~ 0.903，优化版 0.9015），平均后 0.906；
   - HomeCredit：6 个成员都高于优化版联邦，最好的一个（0.7560）与平均（0.7557）相当。

**对报告的意义**：默认组合「I2 + I3 + M6」的第二层，在 7 个代理显著任务中 6 个有真实 SGB 证据，幅度 +0.004 ~ +0.012；代价是联邦计算量约 6 倍。

**边界**：
- 对照（优化版联邦）为 seed 11 单次拟合，区间只含测试集抽样，不含训练随机性；本步任务中只有 mock 重合 50 的对照含列采样随机性，其 6 seed 结果在 S5.M12 另报告；
- 明文直方图只在实验代码里替换一个方法，用于正式联调需经安全评审；
- 公开数据上的开发证据，不换算成汇丰的命中率或收益；技术侧自评，未经法务复核（W-001）。